# Smart Police Patrol v2 — XGBoost + Enhanced Features
**24-hour forecast horizon | H3 res-9 | Walk-forward validation**

Improvements over v1:
- **More lag/rolling features**: lag_21, lag_28, roll_mean_21, roll_std_14, EWM-14
- **Cyclical encodings**: month_sin/cos, extended calendar flags
- **Ring-2 spatial features**: 12-cell second-ring neighbour sum
- **Cell-rank feature**: percentile rank of cell by expanding historical mean
- **Early stopping** inside each fold (last training week = val set)
- **Isotonic recalibration** on walk-forward residuals
- **24h-only** severity & patrol allocation (no 48h blending)
- Loads v1 XGBoost baseline from `model_artifacts/` for comparison


## 0 · Setup & configuration

In [ ]:
import pandas as pd, numpy as np, json, h3, warnings, time
import xgboost as xgb
from sklearn.isotonic import IsotonicRegression
warnings.filterwarnings('ignore')

CSV_PATH           = 'jan to may police violation_anonymized791b166.csv'
H3_RES             = 9
TZ                 = 'Asia/Kolkata'
MIN_CELL_EVENTS    = 50
MIN_TRAIN_WEEKS    = 4     # +1 vs v1: early stopping needs train+val split
TRAIN_WINDOW_WEEKS = None  # expanding window
CALIB_FOLDS        = 8     # walk-forward folds used to fit isotonic calibrator
RANDOM_STATE       = 42
t0 = time.time()


## 1 · Load & clean

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False, usecols=[
    'id','latitude','longitude','created_datetime',
    'police_station','violation_type','vehicle_type'])

df['dt']   = pd.to_datetime(df['created_datetime'], errors='coerce', utc=True).dt.tz_convert(TZ)
df         = df.dropna(subset=['dt','latitude','longitude'])
df         = df[df.latitude.between(12.7, 13.2) & df.longitude.between(77.3, 77.9)]
df['date'] = df['dt'].dt.floor('D').dt.tz_localize(None)
df['cell'] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(df.latitude, df.longitude)]
print(f'rows={len(df):,}  cells={df.cell.nunique()}  '
      f'span={df.date.min().date()}..{df.date.max().date()}  ({round(time.time()-t0,1)}s)')


## 2 · Daily panel + zero-fill + weekly batches

In [ ]:
daily = df.groupby(['cell','date']).size().rename('count').reset_index()

vc   = df.cell.value_counts()
keep = vc[vc >= MIN_CELL_EVENTS].index
print(f'modeled cells: {len(keep)} of {df.cell.nunique()}')

dates = pd.date_range(df.date.min(), df.date.max(), freq='D')
idx   = pd.MultiIndex.from_product([keep, dates], names=['cell','date'])
panel = daily.set_index(['cell','date']).reindex(idx, fill_value=0).reset_index()

stmap = df.groupby('cell')['police_station'].agg(lambda s: s.mode().iloc[0])
panel['station'] = panel['cell'].map(stmap).astype('category')

panel = panel.sort_values(['cell','date']).reset_index(drop=True)
panel['week_idx'] = ((panel.date - panel.date.min()).dt.days // 7).astype(int)
print('panel shape', panel.shape, '| weeks', panel.week_idx.nunique())


## 3 · Target — 24h horizon only

In [ ]:
g = panel.groupby('cell')['count']
panel['y_t1'] = g.shift(-1)   # next-day count: the ONLY prediction target


## 4 · Feature engineering

**4.1 Calendar + cyclical encodings**

In [ ]:
import holidays
ka  = holidays.India(subdiv='KA', years=[2023, 2024])
hol = pd.to_datetime([d for d in ka if panel.date.min() <= pd.Timestamp(d) <= panel.date.max()])

panel['dow']            = panel.date.dt.dayofweek
panel['is_weekend']     = (panel.dow >= 5).astype(int)
panel['month']          = panel.date.dt.month
panel['day_of_month']   = panel.date.dt.day
panel['quarter']        = panel.date.dt.quarter
panel['week_of_period'] = panel.week_idx
panel['is_month_start'] = (panel.day_of_month <= 3).astype(int)
panel['is_month_end']   = (panel.day_of_month >= 28).astype(int)

# cyclical encodings — prevent ordinal discontinuity at boundary
panel['dow_sin']   = np.sin(2*np.pi*panel.dow/7)
panel['dow_cos']   = np.cos(2*np.pi*panel.dow/7)
panel['month_sin'] = np.sin(2*np.pi*(panel.month-1)/12)
panel['month_cos'] = np.cos(2*np.pi*(panel.month-1)/12)

panel['is_holiday']         = panel.date.isin(hol).astype(int)
panel['days_from_holiday']  = panel.date.apply(
    lambda d: (d - hol[hol<=d]).days.min() if (hol<=d).any() else 999)
panel['days_to_holiday']    = panel.date.apply(
    lambda d: (hol[hol>=d] - d).days.min() if (hol>=d).any() else 999)


**4.2 Lags, rolling stats, trend** (all leakage-safe via shift ≥ 1)

In [ ]:
g  = panel.groupby('cell')['count']
gp = panel.groupby('cell')['count']

for L in [1, 2, 3, 7, 14, 21, 28]:
    panel[f'lag_{L}'] = g.shift(L)

def roll(s, w, fn):
    return s.shift(1).rolling(w, min_periods=max(2, w//2)).agg(fn)

panel['roll_mean_3']  = gp.transform(lambda s: roll(s, 3,  'mean'))
panel['roll_mean_7']  = gp.transform(lambda s: roll(s, 7,  'mean'))
panel['roll_mean_14'] = gp.transform(lambda s: roll(s, 14, 'mean'))
panel['roll_mean_21'] = gp.transform(lambda s: roll(s, 21, 'mean'))
panel['roll_std_7']   = gp.transform(lambda s: roll(s, 7,  'std'))
panel['roll_std_14']  = gp.transform(lambda s: roll(s, 14, 'std'))
panel['roll_max_7']   = gp.transform(lambda s: roll(s, 7,  'max'))
panel['roll_median_7']= gp.transform(lambda s: roll(s, 7,  'median'))
panel['ewm_7']        = gp.transform(lambda s: s.shift(1).ewm(span=7,  min_periods=2).mean())
panel['ewm_14']       = gp.transform(lambda s: s.shift(1).ewm(span=14, min_periods=4).mean())

# acceleration & trend
panel['accel']    = panel.roll_mean_3 - panel.roll_mean_7
panel['diff_1']   = panel.lag_1 - panel.lag_2
panel['diff_7']   = panel.lag_7 - panel.lag_14
prevwk = gp.transform(lambda s: roll(s, 7, 'mean').shift(7))
panel['growth_7'] = (panel.roll_mean_7 - prevwk) / (prevwk + 1)
prev4w = gp.transform(lambda s: roll(s, 7, 'mean').shift(28))
panel['growth_28'] = (panel.roll_mean_7 - prev4w) / (prev4w + 1)


**4.3 Same-weekday stats & cell expanding mean**

In [ ]:
def past_dow_mean(d):
    d = d.sort_values('date')
    return d.groupby('dow')['count'].apply(lambda s: s.shift(1).expanding().mean())

panel['same_dow_mean'] = (panel.groupby('cell', group_keys=False)
                               .apply(past_dow_mean).reset_index(level=0, drop=True))
panel['cell_expanding_mean'] = gp.transform(lambda s: s.shift(1).expanding().mean())
panel['dev_from_dow']        = panel.lag_7 - panel.same_dow_mean

# percentile rank of this cell among all cells on each date (how 'hot' vs peers)
panel['cell_rank'] = panel.groupby('date')['cell_expanding_mean'].rank(pct=True)


**4.4 Spatial — ring-1 AND ring-2 neighbour features**

In [ ]:
# ring-1: 6 adjacent cells
nbrs1 = {c: [x for x in h3.grid_disk(c, 1) if x != c] for c in keep}
# ring-2: 12 cells at distance 2 (not in ring-1)
nbrs2 = {c: [x for x in h3.grid_disk(c, 2)
             if x != c and x not in set(nbrs1[c])] for c in keep}

wide = panel.pivot_table(index='date', columns='cell', values='count', fill_value=0)
prev = wide.shift(1)

def ring_sum(nbr_dict, name):
    ns = {}
    for c, nn in nbr_dict.items():
        cols = [x for x in nn if x in prev.columns]
        ns[c] = prev[cols].sum(axis=1) if cols else pd.Series(0, index=prev.index)
    df_ = pd.DataFrame(ns).stack().rename(name).reset_index()
    df_.columns = ['date','cell', name]
    return df_

panel = panel.merge(ring_sum(nbrs1, 'nbr_count_lag1'),  on=['date','cell'], how='left')
panel = panel.merge(ring_sum(nbrs2, 'nbr_ring2_lag1'), on=['date','cell'], how='left')
print(f'spatial features added  ({round(time.time()-t0,1)}s)')


**4.5 Composition shares**

In [ ]:
BUCKET = {'WRONG PARKING':'wrong','NO PARKING':'no_parking',
          'PARKING IN A MAIN ROAD':'road','PARKING NEAR ROAD CROSSING':'road',
          'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS':'road','PARKING ON FOOTPATH':'footpath'}
def primary(x):
    try: t = json.loads(x)
    except: return 'other'
    return BUCKET.get(t[0], 'other') if t else 'other'

df['vbucket'] = df['violation_type'].apply(primary)
df['is2w']    = df['vehicle_type'].isin(['SCOOTER','MOTOR CYCLE','MOPED'])

shares = df.groupby(['cell','date']).agg(
    share_wrong   =('vbucket', lambda s:(s=='wrong').mean()),
    share_nopark  =('vbucket', lambda s:(s=='no_parking').mean()),
    share_2wheeler=('is2w','mean')).reset_index()
panel = panel.merge(shares, on=['cell','date'], how='left')

for col in ['share_wrong','share_nopark','share_2wheeler']:
    panel[col] = (panel.groupby('cell')[col]
                       .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()))

print(f'features built — total cols {panel.shape[1]}  ({round(time.time()-t0,1)}s)')


## 5 · Walk-forward training — XGBoost with early stopping
Each fold: train on all prior weeks, hold out the **last training week** as val for early stopping,
predict the **current week**. Early stopping keeps the best n_estimators per fold automatically.

In [ ]:
FEATS = [
    # calendar
    'dow','is_weekend','month','day_of_month','quarter','week_of_period',
    'is_month_start','is_month_end',
    'dow_sin','dow_cos','month_sin','month_cos',
    'is_holiday','days_to_holiday','days_from_holiday',
    # lags
    'lag_1','lag_2','lag_3','lag_7','lag_14','lag_21','lag_28',
    # rolling
    'roll_mean_3','roll_mean_7','roll_mean_14','roll_mean_21',
    'roll_std_7','roll_std_14','roll_max_7','roll_median_7',
    'ewm_7','ewm_14',
    # trend
    'accel','diff_1','diff_7','growth_7','growth_28',
    # cell-level
    'same_dow_mean','cell_expanding_mean','dev_from_dow','cell_rank',
    # spatial
    'nbr_count_lag1','nbr_ring2_lag1',
    # composition
    'share_wrong','share_nopark','share_2wheeler',
    # categorical
    'station'
]

def xgb_model():
    return xgb.XGBRegressor(
        objective='count:poisson',
        n_estimators=1000,          # ceiling; early stopping picks actual best
        learning_rate=0.03,
        max_depth=5,                # shallower = less overfit on sparse res-9
        min_child_weight=3,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.5,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=40,
        random_state=RANDOM_STATE,
        verbosity=0,
    )

data = panel.dropna(subset=['y_t1']).copy()
data['station'] = data['station'].astype('category')

weeks = sorted(data.week_idx.unique())
preds = []
best_iters = []

for w in weeks:
    if w < MIN_TRAIN_WEEKS:
        continue

    tr = data[data.week_idx < w].copy()
    te = data[data.week_idx == w].copy()
    if len(te) == 0:
        continue

    # align categories
    all_cats = tr['station'].cat.categories.union(te['station'].cat.categories)
    tr['station'] = tr['station'].cat.set_categories(all_cats)
    te['station'] = te['station'].cat.set_categories(all_cats)

    # hold out last training week as val for early stopping
    val_w = tr.week_idx.max()
    val   = tr[tr.week_idx == val_w].copy()
    tr2   = tr[tr.week_idx <  val_w].copy()
    val['station'] = val['station'].cat.set_categories(all_cats)
    tr2['station'] = tr2['station'].cat.set_categories(all_cats)

    m = xgb_model()
    if len(tr2) > 0:
        m.fit(tr2[FEATS], tr2['y_t1'],
              eval_set=[(val[FEATS], val['y_t1'])],
              verbose=False)
    else:
        m.fit(tr[FEATS], tr['y_t1'])

    best_iters.append(m.best_iteration if hasattr(m, 'best_iteration') else None)
    out = te[['cell','date','week_idx','y_t1','lag_7','cell_expanding_mean']].copy()
    out['pred']  = np.clip(m.predict(te[FEATS]), 0, None)
    out['naive'] = out['lag_7'].fillna(out['cell_expanding_mean']).fillna(0)
    preds.append(out)

preds = pd.concat(preds, ignore_index=True)
valid_iters = [x for x in best_iters if x is not None]
print(f'walk-forward done: {preds.week_idx.nunique()} batches, {len(preds):,} predictions')
print(f'early stopping — best_iter avg={np.mean(valid_iters):.0f}, '
      f'min={min(valid_iters)}, max={max(valid_iters)}')


## 6 · Isotonic recalibration
Fit isotonic regression on the first `CALIB_FOLDS` walk-forward folds, apply to the rest.
Corrects systematic bias in predicted counts without touching the ranking.

In [ ]:
fold_weeks = sorted(preds.week_idx.unique())
calib_weeks = fold_weeks[:CALIB_FOLDS]
eval_weeks  = fold_weeks[CALIB_FOLDS:]

calib_df = preds[preds.week_idx.isin(calib_weeks)]
eval_df  = preds[preds.week_idx.isin(eval_weeks)].copy()

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(calib_df['pred'].values, calib_df['y_t1'].values)

eval_df['pred_cal'] = np.clip(iso.predict(eval_df['pred'].values), 0, None)
eval_df['pred_raw'] = eval_df['pred']         # keep raw for comparison
eval_df['pred']     = eval_df['pred_cal']

print(f'calibration: {len(calib_weeks)} folds fit, {len(eval_weeks)} folds evaluated')
print(f'  raw  MAE = {np.abs(eval_df.pred_raw - eval_df.y_t1).mean():.3f}')
print(f'  cal  MAE = {np.abs(eval_df.pred_cal - eval_df.y_t1).mean():.3f}')


## 7 · Evaluation — full walk-forward + calibrated eval window

In [ ]:
def poisson_dev(y, p):
    p = np.clip(p, 1e-6, None)
    return 2*np.mean(np.where(y>0, y*np.log(y/p), 0) - (y-p))

def precision_at_k(frame, k=10):
    hits = []
    for _, g in frame.groupby('date'):
        if len(g) < k: continue
        hits.append(len(set(g.nlargest(k,'y_t1').cell) & set(g.nlargest(k,'pred').cell))/k)
    return np.mean(hits)

y, p, nv = preds.y_t1.values, preds.pred.values, preds.naive.values
print('=== Full walk-forward (all folds) ===')
print('            model     naive')
print(f'MAE      {np.abs(p-y).mean():8.3f}  {np.abs(nv-y).mean():8.3f}')
print(f'RMSE     {np.sqrt(((p-y)**2).mean()):8.3f}  {np.sqrt(((nv-y)**2).mean()):8.3f}')
print(f'PoisDev  {poisson_dev(y,p):8.3f}  {poisson_dev(y,nv):8.3f}')
print(f'P@10     {precision_at_k(preds,10):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive),10):8.3f}')
print(f'P@25     {precision_at_k(preds,25):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive),25):8.3f}')

if len(eval_weeks) > 0:
    ey, ep = eval_df.y_t1.values, eval_df.pred.values
    print(f'\n=== Calibrated eval folds ({len(eval_weeks)} weeks) ===')
    print(f'MAE  {np.abs(ep-ey).mean():.3f}  |  P@10  {precision_at_k(eval_df,10):.3f}')

wk = (preds.assign(ae=np.abs(preds.pred-preds.y_t1),
                   ae_n=np.abs(preds.naive-preds.y_t1))
           .groupby('week_idx').agg(n=('y_t1','size'),
                                    mae_model=('ae','mean'),
                                    mae_naive=('ae_n','mean')).round(3))
wk


## 8 · Severity engine — 24h only

In [ ]:
latest = preds[preds.week_idx == preds.week_idx.max()].copy()

# severity calibrated on full walk-forward distribution (24h counts only)
q = preds.y_t1.quantile([0.50, 0.80, 0.95]).values
def severity(c):
    if c <= q[0]: return 'Low'
    if c <= q[1]: return 'Medium'
    if c <= q[2]: return 'High'
    return 'Critical'
OFFICERS = {'Low':1, 'Medium':2, 'High':3, 'Critical':4}

latest['severity'] = latest['pred'].apply(severity)
latest['officers'] = latest['severity'].map(OFFICERS)

latest = latest.merge(
    panel[['cell','date','growth_7','nbr_count_lag1','nbr_ring2_lag1']],
    on=['cell','date'], how='left')

# risk score: 24h pred × trend factor × ring-1 pressure × ring-2 pressure
nbr1_mean = latest['nbr_count_lag1'].mean() + 1
nbr2_mean = latest['nbr_ring2_lag1'].mean() + 1
latest['risk_score'] = (
    latest['pred']
    * (1 + latest['growth_7'].fillna(0).clip(-0.9, None))
    * (1 + latest['nbr_count_lag1'].fillna(0) / nbr1_mean)
    * (1 + 0.3 * latest['nbr_ring2_lag1'].fillna(0) / nbr2_mean)  # ring-2 at 0.3x weight
)

print('severity cutoffs (p50/p80/p95):', q.round(2))
print(f'\nTop 10 hotspots for {latest.date.max().date()} (24h forecast)')
latest.sort_values('risk_score', ascending=False).head(10)[
    ['cell','pred','severity','officers','risk_score']].round(2)


## 9 · Save model + outputs

In [ ]:
import joblib, os
os.makedirs('model_artifacts', exist_ok=True)

# save final model (trained on everything except last val week of last fold)
m.save_model('model_artifacts/xgb_patrol_v2.json')
joblib.dump({
    'feats': FEATS,
    'station_cats': all_cats,
    'h3_res': H3_RES,
    'severity_quantiles': q.tolist(),
    'isotonic_calibrator': iso,
    'best_iter_avg': float(np.mean(valid_iters)) if valid_iters else None
}, 'model_artifacts/model_v2_meta.pkl')

out_cols = ['cell','date','pred','severity','officers','risk_score']
latest.sort_values('risk_score', ascending=False)[out_cols].round(3).to_csv(
    'patrol_forecast_v2_latest.csv', index=False)
try:
    preds.to_parquet('walkforward_v2_predictions.parquet', index=False)
    print('saved: patrol_forecast_v2_latest.csv, walkforward_v2_predictions.parquet')
except Exception:
    preds.to_csv('walkforward_v2_predictions.csv', index=False)
    print('saved: patrol_forecast_v2_latest.csv, walkforward_v2_predictions.csv')
print('model saved: model_artifacts/xgb_patrol_v2.json + model_v2_meta.pkl')


## 10 · Unseen-test validation map (last 4 weeks)

In [ ]:
TEST_WEEKS = 4

def build_test_map(preds, test_weeks=TEST_WEEKS, out_html='patrol_map_v2_test.html',
                   center=(12.9716, 77.5946), zoom=11):
    wmax = preds.week_idx.max()
    test = preds[preds.week_idx >= wmax - test_weeks + 1].copy()
    d0, d1 = pd.to_datetime(test.date).min(), pd.to_datetime(test.date).max()
    agg = test.groupby('cell').agg(pred=('pred','sum'), actual=('y_t1','sum')).reset_index()
    agg['err'] = agg.pred - agg.actual

    mae = np.abs(test.pred - test.y_t1).mean()
    def p_at_k(frame, k=10):
        hits=[]
        for _, g in frame.groupby('date'):
            if len(g) < k: continue
            hits.append(len(set(g.nlargest(k,'y_t1').cell) & set(g.nlargest(k,'pred').cell))/k)
        return np.mean(hits)
    p10 = p_at_k(test, 10)

    qs   = np.quantile(agg.actual, [0,.2,.4,.6,.8,1.0])
    RAMP = ['#fee5d9','#fcae91','#fb6a4a','#de2d26','#a50f15']
    def color(v):
        for i in range(5):
            if v <= qs[i+1]: return RAMP[i]
        return RAMP[-1]
    def layer(valcol):
        feats=[]
        for _, r in agg.iterrows():
            b    = h3.cell_to_boundary(r['cell'])
            ring = [[lng,lat] for lat,lng in b] + [[b[0][1], b[0][0]]]
            feats.append({"type":"Feature","properties":{
                "cell":r['cell'],"pred":round(float(r.pred),1),
                "actual":round(float(r.actual),1),"err":round(float(r.err),1),
                "color":color(r[valcol])},
                "geometry":{"type":"Polygon","coordinates":[ring]}})
        return {"type":"FeatureCollection","features":feats}
    gj_pred, gj_act = layer('pred'), layer('actual')
    legend = "".join(f'<div><span style="background:{c}"></span>{int(qs[i])}-{int(qs[i+1])}</div>'
                     for i, c in enumerate(RAMP))
    html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>v2 Unseen Test</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 html,body,#map{{height:100%;margin:0}}
 .panel{{position:absolute;top:10px;left:50px;z-index:1000;background:#fff;padding:8px 12px;
   border-radius:8px;font:13px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3);max-width:380px}}
 .legend{{position:absolute;bottom:20px;right:12px;z-index:1000;background:#fff;padding:10px 12px;
   border-radius:8px;font:12px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3)}}
 .legend div{{display:flex;align-items:center;gap:6px;margin:2px 0}}
 .legend span{{width:14px;height:14px;border-radius:3px;display:inline-block}}
</style></head><body>
<div id="map"></div>
<div class="panel"><b>v2 Unseen test — 24h XGBoost</b><br>
 {d0.date()} -> {d1.date()} ({test_weeks} weeks)<br>
 MAE {mae:.2f} · Precision@10 {p10:.0%}<br>
 <small>Toggle Predicted/Actual (top-right). Same scale -> matching colours = good forecast.</small></div>
<div class="legend"><b>Violations / window</b>{legend}</div>
<script>
const map=L.map('map').setView([{center[0]},{center[1]}],{zoom});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
  {{maxZoom:19,attribution:'(c) OpenStreetMap'}}).addTo(map);
function mk(data){{return L.geoJSON(data,{{
  style:f=>({{color:'#333',weight:1,fillColor:f.properties.color,fillOpacity:0.6}}),
  onEachFeature:(f,l)=>{{const p=f.properties;
    l.bindPopup(`<b>${{p.cell}}</b><br>Predicted: ${{p.pred}}<br>Actual: ${{p.actual}}<br>Error: ${{p.err}}`);}}}})
}}}}
const predL=mk({json.dumps(gj_pred)}), actL=mk({json.dumps(gj_act)});
actL.addTo(map);
L.control.layers(null,{{"Actual (truth)":actL,"Predicted (model)":predL}},{{collapsed:false}}).addTo(map);
</script></body></html>"""
    with open(out_html,'w', encoding='utf-8') as f: f.write(html)
    print(f'v2 unseen test: {d0.date()}..{d1.date()} | {len(agg)} cells | MAE {mae:.2f} | P@10 {p10:.0%} -> {out_html}')
    return out_html

build_test_map(preds, TEST_WEEKS, 'patrol_map_v2_test.html')
from IPython.display import IFrame
IFrame('patrol_map_v2_test.html', width='100%', height=560)


## 11 · Feature importance

In [ ]:
import matplotlib.pyplot as plt
fi = pd.Series(m.feature_importances_, index=FEATS).sort_values(ascending=False)
print('Top 20 features:')
print(fi.head(20).round(4).to_string())
fig, ax = plt.subplots(figsize=(8,6))
fi.head(20).sort_values().plot.barh(ax=ax)
ax.set_title('XGBoost v2 — feature importance (gain)')
ax.set_xlabel('importance')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=120)
plt.show()
print('saved: feature_importance_v2.png')
